# Day 1 Demo — Calling an LLM API from Python

## What we will build

We will create a small **website summarizer** and use it to learn the basic workflow of an LLM application.

Learning path:

1. Install and import libraries
2. Load an API key securely
3. Create an API client
4. Send a simple message to an LLM
5. Understand `system` and `user` messages
6. Build prompts with Python
7. Read website content
8. Ask the LLM to summarize it
9. Display the answer as Markdown
10. Adapt the same pattern to a business example

The overall idea is:

**Python application → API request → LLM → response → Python displays the result**

## 1. Install the required libraries

Run this cell if these packages are not already installed in your notebook environment.

- `openai` — Python client for an OpenAI-compatible API (Ollama exposes this same API locally)
- `python-dotenv` — loads values from a `.env` file
- `requests` — downloads webpage HTML
- `beautifulsoup4` — extracts readable text from HTML
- `IPython` — improves notebook output

You will also need [Ollama](https://ollama.com) installed and running locally, with the `phi3` model pulled:

```bash
ollama pull phi3
```


In [ ]:
%pip install openai python-dotenv requests beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

html = """
<html>
    <head>
        <title>My AI Website</title>

        <style>
            body {
                background-color: blue;
            }

            h1 {
                color: red;
            }
        </style>

        <script>
            console.log("This is JavaScript");
            alert("Hello!");
        </script>
    </head>

    <body>
        <h1>Welcome to AI World<b>and agentic world</b></h1>

        <p>AI helps computers perform intelligent tasks.</p>

        <p>Machine Learning is a part of AI.</p>

        <noscript>
            Please enable JavaScript.
        </noscript>
    </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")

print("BEFORE REMOVING")
print(soup)

print("\nSTYLE AND SCRIPT FOUND")
print(soup(["style", "script"]))

for element in soup(["script", "style", "noscript"]):
    element.decompose()

print("\nAFTER REMOVING")
print(soup)

print("\nTEXT WITHOUT SEPARATOR AND STRIP")
print(soup.get_text())

text = soup.get_text(separator=" ", strip=True)
print("\nONLY TEXT")
print(text)

## 2. Import the libraries

Imports make functionality from external packages available to Python.

The website-reading function below is included directly in this notebook so the demo is self-contained.

In [ ]:
import os

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


def fetch_website_contents(url):
    """Download a webpage and return its visible text."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"}

    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
   

    for element in soup(["script", "style", "noscript"]):
        element.decompose()

    return soup.get_text(separator=" ", strip=True)

In [ ]:
fetch_website_contents("https://uudoon.in/")

## 3. Load the API key securely

Ollama runs locally, so there is no secret API key to protect — the model runs on your own machine.

The OpenAI-compatible client still requires an `api_key` value to be passed, but for Ollama this can be any placeholder string.

We keep the same `.env` pattern for consistency with other providers:

```text
OLLAMA_API_KEY=ollama
```

The notebook reads this value using `python-dotenv`.


In [ ]:
load_dotenv()

api_key = os.getenv("OLLAMA_API_KEY", "ollama")

if not api_key:
    print("No API key was found. Check your .env file.")
else:
    print("API key loaded successfully.")


## 4. Create the API client

The client is the Python object that communicates with the model provider.

This demo uses an OpenAI-compatible endpoint hosted locally by Ollama. The endpoint and model can be changed when using another compatible provider.


In [ ]:
openai = OpenAI(
    api_key=os.getenv("OLLAMA_API_KEY", "ollama"),
    base_url="http://localhost:11434/v1",
)


## 5. Make the first LLM request

Before building the website summarizer, start with a very small request.

A chat request is represented by a list of messages. Each message has a `role` and `content`.

In [ ]:
message = "What is 2 + 2?"

messages = [
    {
        "role": "user",
        "content": message
    }
]

response = openai.chat.completions.create(
    model="qwen3:1.7b",
    messages=messages
)
print(response.choices[0].message)
print(response.choices[0].message.content)


## 6. Understand the two important message roles

A common LLM application uses:

- **`system`** — describes the assistant's behavior, role, tone, or output requirements.
- **`user`** — contains the request and the information the model should work with.

A simple way to remember it:

**System = how to behave**

**User = what to do**

In [ ]:
system_prompt = """
You are a helpful website summarization assistant.
Create concise summaries using clear Markdown.
Focus on useful information and ignore menus or navigation text.
"""

user_prompt = """
Summarize the following website content in a few bullet points.
If there are recent announcements or important updates, mention them.

Website content:
[website text will be inserted here]
"""

## 7. Build messages with a function

Instead of creating the message list manually for every website, create a reusable function.

This separates **prompt construction** from the rest of the application.

In [ ]:
def messages_for(website):
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt.replace(
                "[website text will be inserted here]",
                website
            )
        }
    ]

## 8. Inspect what will be sent to the model

During development, inspecting intermediate values helps students understand the data flowing through the program.

In [ ]:
sample_website_text = (
    "Example website text about an AI company, its products, "
    "and recent announcements."
)

messages = messages_for(sample_website_text)

messages

## 9. Connect web scraping and the LLM

Now combine the main steps:

1. Download the webpage
2. Extract its text
3. Build the messages
4. Send them to the model
5. Return the generated summary

In [ ]:
def summarize(url):
    website = fetch_website_contents(url)

    response = openai.chat.completions.create(
        model="qwen3:1.7b",
        messages=messages_for(website)
    )

    return response.choices[0].message.content


## 10. Run the complete workflow

This cell uses the complete pipeline on a public website.

A simple HTML scraper may not work with every website. Sites that depend heavily on JavaScript or block automated requests can require browser automation or another retrieval method.

In [ ]:
summary = summarize("https://uudoon.in")
print(summary)

## 11. Display the answer as Markdown

LLMs frequently return Markdown. Jupyter can render that Markdown directly, which makes the result easier to read.

In [ ]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://uudoon.in")

## 12. Reuse the same function

The summarization function is reusable. Change the URL and the same application can process another webpage.

In [ ]:
display_summary("https://anthropic.com")

## 13. Website limitations

The simple `requests + BeautifulSoup` approach works best for webpages whose useful content is present in the downloaded HTML.

Some modern websites render their content with JavaScript. In those cases, a browser automation tool such as Selenium or Playwright can be used to load the page before extracting its text.

For this first demo, keep the workflow simple and focus on understanding the LLM API.

In [ ]:
# Try another simple public webpage:
# display_summary("https://example.com")

## 14. Business example — summarize a customer email

The same LLM pattern can be used without web scraping.

Here we provide a customer email and ask the model to produce a useful support summary.

In [ ]:
email_text = """
Hello team,

I purchased the annual subscription last week, but my account is still
showing the free plan. I have already been charged, and I would like the
subscription activated or the payment refunded.

Thanks.
"""

system_prompt = """
You are a professional customer-support assistant.
Analyze the email and respond in Markdown with:
1. A one-line summary
2. The customer's main issue
3. The requested action
4. A short suggested email subject
"""

user_prompt = f"""
Analyze this customer email:

{email_text}
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

response = openai.chat.completions.create(
    model="qwen3:1.7b",
    messages=messages
)

display(Markdown(response.choices[0].message.content))


## 15. What students should remember

### Python concepts
- Imports
- Variables
- Functions
- Lists
- Dictionaries
- Environment variables

### LLM concepts
- API client
- Model
- Messages
- `system` role
- `user` role
- Prompt
- API response
- Generated text

### Application flow

```text
Website URL
    ↓
Python downloads webpage
    ↓
Text is extracted
    ↓
Python creates messages
    ↓
LLM API receives the request
    ↓
LLM generates a summary
    ↓
Python displays the result
```

The important step is moving from **asking an LLM a simple question** to **using an LLM as a component inside a real Python application**.

## 16. Student exercise

Modify the prompts and build one of these:

1. News summarizer
2. Product-page summarizer
3. Email subject generator
4. Meeting-note summarizer
5. Website audience detector

### Challenge

First change only the prompts, not the Python workflow.

This demonstrates how much application behavior can be changed through prompt design.